# Solar Inverter Failure Prediction Pipeline (Colab Version)

This notebook implements the complete machine learning pipeline for solar inverter failure prediction using internal telemetry data.

### Features:
1. **Full Feature Space**: Support for all 71+ hardware telemetry signals.
2. **Data Cleaning**: Robust handling of invalid sensor readings and mission values.
3. **Feature Engineering**: Automated generation of efficiency, electrical loss, and time-based features.
4. **Dual Modeling**: XGBoost for failure risk and Isolation Forest for anomaly detection.
5. **Integration Ready**: Saves artifacts compatible with the production backend API.

In [ ]:
# Install dependencies
!pip install xgboost joblib pandas numpy scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import IsolationForest
import json
import os
import joblib
from google.colab import files

EXCLUDE_COLS = ['_id', 'mac', 'timestamp', 'timestampDate', 'createdAt', 'fromServer', 
                'dataLoggerModelId', '__v', 'inverters[0].serial', 'meters[0].serial', 
                'inverters[0].id', 'meters[0].id', 'smu[0].id']

## 1. Data Loading
Upload your `Copy of 54-10-EC-8C-14-69.raws.csv` or similar telemetry dataset.

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
print(f"Loaded {file_path}")

In [ ]:
def clean_data(df):
    if "dc_voltage" in df.columns:
        df = df[df["dc_voltage"] > 0]
    if "ac_power" in df.columns:
        df = df[df["ac_power"] >= 0]
    if "grid_frequency" in df.columns:
        df = df[(df["grid_frequency"] >= 45) & (df["grid_frequency"] <= 65)]
    
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    for col in numerical_cols:
        df[col] = df[col].fillna(df[col].median())
            
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp')
            
    return df

def feature_engineering(df):
    if "dc_voltage" in df.columns and "dc_current" in df.columns:
        df["dc_power"] = df["dc_voltage"] * df["dc_current"]
    
    if "dc_power" in df.columns and "ac_power" in df.columns:
        df["power_loss"] = df["dc_power"] - df["ac_power"]
        df["efficiency"] = np.where(df["dc_power"] > 0, df["ac_power"] / df["dc_power"], 0)
        df["efficiency"] = df["efficiency"].clip(0, 1.1)

    if "inverter_temp" in df.columns:
        df["thermal_stress"] = df["inverter_temp"]

    if 'timestamp' in df.columns:
        df['hour'] = df['timestamp'].dt.hour
        df['day_of_week'] = df['timestamp'].dt.dayofweek
        df['month'] = df['timestamp'].dt.month

    return df

In [ ]:
def generate_target_labels(df):
    temp = df.get('inverter_temp', 0)
    eff = df.get('efficiency', 1.0)
    dc_p = df.get('dc_power', 0)
    alarm = df.get('alarm_code', 0)
    
    failure_conditions = (
        (temp > 85) |  
        ((eff < 0.7) & (dc_p > 500)) | 
        (alarm != 0) 
    )
    df['failure_risk'] = failure_conditions.astype(int)
    return df

In [ ]:
print("Preprocessing data...")
df_raw = pd.read_csv(file_path)

feature_map = {
    'inverters[0].pv1_power': 'ac_power',
    'inverters[0].pv1_voltage': 'dc_voltage',
    'inverters[0].pv1_current': 'dc_current',
    'inverters[0].temp': 'inverter_temp',
    'meters[0].v_r': 'grid_voltage',
    'meters[0].freq': 'grid_frequency',
    'meters[0].pf': 'power_factor',
    'inverters[0].kwh_total': 'kwh_total',
    'inverters[0].kwh_today': 'kwh_today',
    'inverters[0].alarm_code': 'alarm_code'
}

numeric_df = df_raw.select_dtypes(include=[np.number])
original_cols = [col for col in numeric_df.columns if col not in EXCLUDE_COLS]
df = df_raw[original_cols + (['timestamp'] if 'timestamp' in df_raw.columns else [])].copy()
df = df.rename(columns={k: v for k, v in feature_map.items() if k in df.columns})

actual_original_names = [feature_map.get(col, col) for col in original_cols]

for col in [c for c in df.columns if c != 'timestamp']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = clean_data(df)
df = feature_engineering(df)
df = generate_target_labels(df)

def sanitize_name(name):
    return name.replace('[', '_').replace(']', '_').replace('<', '_').replace('>', '_')
df.columns = [sanitize_name(c) for c in df.columns]
actual_original_names = [sanitize_name(c) for c in actual_original_names]

X = df.drop(columns=['failure_risk', 'timestamp', 'alarm_code'], errors='ignore')
y = df['failure_risk']
print(f"Dataset matched: {X.shape[0]} samples, {X.shape[1]} features.")

In [ ]:
print("Training XGBoost...")
model = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, objective='binary:logistic', random_state=42)
tscv = TimeSeriesSplit(n_splits=5)
for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Training Complete. Final Accuracy: {accuracy:.4f}")

In [ ]:
iso_forest = IsolationForest(contamination=0.01, random_state=42)
iso_forest.fit(X)

model.save_model('solar_failure_model.json')
joblib.dump(iso_forest, 'anomaly_model.pkl')

metadata = {
    'original_features': [c for c in X.columns if c in actual_original_names],
    'engineered_features': [c for c in X.columns if c not in actual_original_names],
    'metrics': {'accuracy': accuracy}
}
with open('feature_meta.json', 'w') as f:
    json.dump(metadata, f, indent=4)

files.download('solar_failure_model.json')
files.download('anomaly_model.pkl')
files.download('feature_meta.json')